In [9]:
# Basic libraries
import time
import torch
import pandas as pd
from datasets import load_dataset

# Transformers library for LLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Utilities and metrics
from memory_profiler import memory_usage
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
    print(f"Device index: {torch.cuda.current_device()}")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

print(f"Device: {device}")

Using device: NVIDIA GeForce RTX 3060 Ti
Device index: 0
Device: cuda


# Quantization setup

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# LLM setup

In [12]:
model_name = 'meta-llama/Llama-3.2-3B-Instruct'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(model.hf_device_map)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

{'': 0}


# Defining a classification function

In [13]:
def classify(prompt, labels):
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions. Only respond with the label that best describe the text. If the text does not fit any of the labels, respond with 'none'."},
        {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis. The possible labels are: {labels}. Text: {prompt}"}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    num_input_tokens = model_inputs.input_ids.size(1)

    max_label_length = max([len(tokenizer.encode(label, add_special_tokens=False)) for label in labels])

    generated_ids = model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=max_label_length,
        temperature=0.0,
        do_sample=False,
        top_k=0,
        top_p=1.0,
        num_beams=1
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    num_output_tokens = generated_ids[0].size(0)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

    response = response.strip().lower()

    if response not in labels:
        response = "none"

    max_vram_usage = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None


    return [response, num_input_tokens, num_output_tokens, max_vram_usage]

# Loading test dataset

#### Only testing is necessary as the model is pre-trained, so no training/tuning is necessary.

In [14]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


# Preprocessing the dataset

In [15]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()
labels

array(['positive', 'negative'], dtype=object)

# Classifying dataset

In [16]:
columns = ['text', 'true_label', 'predicted_label', 'peak_memory_usage', 'max_vram_usage', 'inference_time', 'num_input_tokens', 'num_output_tokens']

results = pd.DataFrame(columns=columns)
results

,text,true_label,predicted_label,peak_memory_usage,max_vram_usage,inference_time,num_input_tokens,num_output_tokens


In [17]:
for index, row in test.iterrows():
    text = row['text']
    print(f"Text: {text}")


    start = time.perf_counter()
    peak_memory_usage, response = memory_usage((classify, (text, labels),), max_usage=True, retval=True)
    total_time = time.perf_counter() - start

    generated_text = response[0]
    num_input_tokens = response[1]
    num_output_tokens = response[2]
    max_vram_usage = response[3]
    print(f"Response: {generated_text}")
    print(f"Input tokens: {num_input_tokens} || Output tokens: {num_output_tokens}")
    print(f"Peak memory Usage: {peak_memory_usage}")
    print(f"Max VRAM Usage: {max_vram_usage}")
    print(f"Total Time: {total_time}")

    # concat to results
    result = {
        'text': text,
        'true_label': row['label'],
        'predicted_label': generated_text,
        'peak_memory_usage': peak_memory_usage,
        'max_vram_usage': max_vram_usage,
        'inference_time': total_time,
        'num_input_tokens': num_input_tokens,
        'num_output_tokens': num_output_tokens
    }

    results = pd.concat([results, pd.DataFrame([result], columns=columns)], ignore_index=True)
    print("-" * 75)

Text: lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\generation\configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\generation\configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\tra

positive
Response: positive
Input tokens: 154 || Output tokens: 1
Peak memory Usage: 1464.6171875
Max VRAM Usage: 2366.41845703125
Total Time: 2.028370000072755
---------------------------------------------------------------------------
Text: consistently clever and suspenseful .


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


positive


...

negative
Response: negative
Input tokens: 140 || Output tokens: 1
Peak memory Usage: 1272.18359375
Max VRAM Usage: 2364.11474609375
Total Time: 1.1481567999580875
---------------------------------------------------------------------------
Text: enigma is well-made , but it's just too dry and too placid .


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


negative


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


negative
Response: negative
Input tokens: 145 || Output tokens: 1
Peak memory Usage: 1272.18359375
Max VRAM Usage: 2364.93798828125
Total Time: 1.1496895999880508
---------------------------------------------------------------------------


In [18]:
results.to_csv("results/results_llama_ZS_binary2.csv", index=False)
results

,text,true_label,predicted_label,peak_memory_usage,max_vram_usage,inference_time,num_input_tokens,num_output_tokens
0,lovingly photographed in the manner of a golde...,positive,positive,1464.617188,2366.418457,2.028370,154,1
1,consistently clever and suspenseful .,positive,positive,1472.929688,2363.127441,1.354463,134,1
2,"it's like a "" big chill "" reunion of the baade...",positive,positive,1473.054688,2367.076660,1.300386,158,1
3,the story gives ample opportunity for large-sc...,positive,positive,1473.097656,2366.089355,1.287321,152,1
4,"red dragon "" never cuts corners .",positive,positive,1473.125000,2363.292480,1.317719,135,1
...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative,1272.183594,2363.950684,1.199526,139,1
1062,there are many definitions of 'time waster' bu...,negative,negative,1272.183594,2365.431152,1.143334,148,1
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative,1272.183594,2370.038574,1.124101,176,1
1064,the thing looks like a made-for-home-video qui...,negative,negative,1272.183594,2364.114746,1.148157,140,1


In [19]:
print(results['predicted_label'].value_counts())
print(results['true_label'].value_counts())

predicted_label
positive    564
negative    494
none          8
Name: count, dtype: int64
true_label
positive    533
negative    533
Name: count, dtype: int64


In [27]:
y = results['true_label']
y_pred = results['predicted_label']

accuracy = accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred, average='weighted')
recall = recall_score(y, y_pred, average='weighted')
precision = precision_score(y, y_pred, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.8320825515947468
F1 Score: 0.8349752667050706
Recall: 0.8320825515947468
Precision: 0.8397364114049444


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
